# GBM Timezone Audit

**Objective:** Reproduce and verify the timestamp correction documented in T39.

The original GBM predictions file contains timestamps labelled as UTC, although they represent Madrid local time. This can cause incorrect alignment between GBM and Chronos predictions.

This notebook independently verifies the existing correction and generates a properly aligned UTC prediction file.

**Source:** T39 – `hybrid_blend_learned_weights.ipynb`.

**Note:** The original timezone issue was identified in T39. This notebook reproduces and validates that correction; it does not claim to have discovered the issue.

In [ ]:

from google.colab import files
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

uploaded = files.upload()


Saving 2026G3-main.zip to 2026G3-main.zip


In [ ]:

zip_name = next(
    name for name in uploaded
    if name.lower().endswith(".zip")
)

extract_dir = Path("/content/tbda_timezone")

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_dir)

project_dir = extract_dir / "2026G3-main"

print("Cartella progetto:", project_dir.exists())
print("File GBM:", (
    project_dir /
    "predictions_and_eval/All_Zones_GBM_predictions.csv"
).exists())
print("Dataset pulito:", (
project_dir / "cedint_f1_hourly_clean.csv.gz"
).exists())


Cartella progetto: True
File GBM: True
Dataset pulito: True


In [ ]:

import pandas as pd
import numpy as np

gbm_path = project_dir / "predictions_and_eval/All_Zones_GBM_predictions.csv"
clean_path = project_dir / "cedint_f1_hourly_clean.csv.gz"

gbm = pd.read_csv(gbm_path)
clean = pd.read_csv(clean_path, low_memory=False)

print("GBM predictions:", len(gbm))
print("Clean dataset:", len(clean))


GBM predictions: 48221
Clean dataset: 470680


In [ ]:

reference = clean[
    ["timestamp_utc", "timestamp_local",
     "zone", "load_type", "energy_wh_model"]
].copy()

for col in ["timestamp_utc", "timestamp_local"]:
    reference[col] = pd.to_datetime(reference[col])

gbm_check = gbm.copy()
gbm_check["timestamp_utc"] = pd.to_datetime(
    gbm_check["timestamp_utc"]
)

check_utc = gbm_check.merge(
    reference,
    on=["timestamp_utc", "zone", "load_type"],
    how="left"
)

check_local = gbm_check.merge(
    reference,
    left_on=["timestamp_utc", "zone", "load_type"],
    right_on=["timestamp_local", "zone", "load_type"],
    how="left"
)

def compare_values(data):
    valid = data[["y_true", "energy_wh_model"]].dropna()

    matches = np.isclose(
        valid["y_true"],
        valid["energy_wh_model"],
        atol=0.01
    )

    return {
        "rows_compared": len(valid),
        "matches": int(matches.sum()),
        "match_percentage": round(100 * matches.mean(), 2)
    }

print("A - GBM interpreted as UTC:")
print(compare_values(check_utc))

print("\nB - GBM interpreted as Madrid local time:")
print(compare_values(check_local))


A - GBM interpreted as UTC:
{'rows_compared': 48204, 'matches': 2547, 'match_percentage': np.float64(5.28)}

B - GBM interpreted as Madrid local time:
{'rows_compared': 48241, 'matches': 48223, 'match_percentage': np.float64(99.96)}


In [ ]:

# Convertiamo gli orari locali in UTC
gbm_fixed = gbm.copy()

local_times = pd.to_datetime(
    gbm_fixed["timestamp_utc"]
)

gbm_fixed["timestamp_utc"] = (
    local_times
    .dt.tz_localize(
        "Europe/Madrid",
        ambiguous="NaT",
        nonexistent="NaT"
    )
    .dt.tz_convert("UTC")
    .dt.tz_localize(None)
)

print("Original rows:", len(gbm))
print("Converted timestamps:",
      gbm_fixed["timestamp_utc"].notna().sum())

print("Unresolved timestamps:",
      gbm_fixed["timestamp_utc"].isna().sum())


Original rows: 48221
Converted timestamps: 48201
Unresolved timestamps: 20


In [ ]:

# Individuiamo i timestamp non risolti
unresolved = gbm.loc[
    gbm_fixed["timestamp_utc"].isna()
].copy()

print("Unresolved rows:", len(unresolved))

print("\nDates involved:")
print(
    unresolved["timestamp_utc"]
    .str[:10]
    .value_counts()
)

print("\nRows to investigate:")
display(
    unresolved[
        ["timestamp_utc", "zone", "load_type", "y_true"]
    ]
)


Unresolved rows: 20

Dates involved:
timestamp_utc
2022-10-30    20
Name: count, dtype: int64

Rows to investigate:


,timestamp_utc,zone,load_type,y_true
2393,2022-10-30 02:00:00,NW,HVAC,33.5925
2394,2022-10-30 02:00:00,NW,HVAC,33.8550
8019,2022-10-30 02:00:00,NW,Lighting,1.4385
8020,2022-10-30 02:00:00,NW,Lighting,1.1385
14407,2022-10-30 02:00:00,NW,Plugs & Sockets,1431.7830
14408,2022-10-30 02:00:00,NW,Plugs & Sockets,1404.9005
20796,2022-10-30 02:00:00,NE,HVAC,2.6875
20797,2022-10-30 02:00:00,NE,HVAC,2.5670
27210,2022-10-30 02:00:00,NE,Plugs & Sockets,24.1415
27211,2022-10-30 02:00:00,NE,Plugs & Sockets,23.5450


In [ ]:

# Confronto delle 20 righe con il dataset originale

reference_ambiguous = clean[
    (clean["timestamp_local"] == "2022-10-30 02:00:00")
][
    ["timestamp_utc", "timestamp_local",
     "zone", "load_type", "energy_wh_model"]
].copy()

candidates = unresolved.merge(
    reference_ambiguous,
    on=["zone", "load_type"],
    how="left",
    suffixes=("_gbm", "_reference")
)

candidates["value_match"] = np.isclose(
    candidates["y_true"],
    candidates["energy_wh_model"],
    atol=0.01,
    rtol=0
)

print("Candidate matches:")
display(
    candidates[
        ["zone", "load_type", "y_true",
         "energy_wh_model", "timestamp_utc_reference",
         "value_match"]
    ]
)


Candidate matches:


,zone,load_type,y_true,energy_wh_model,timestamp_utc_reference,value_match
0,NW,HVAC,33.5925,33.5925,2022-10-30 00:00:00,True
1,NW,HVAC,33.5925,33.8550,2022-10-30 01:00:00,False
2,NW,HVAC,33.8550,33.5925,2022-10-30 00:00:00,False
3,NW,HVAC,33.8550,33.8550,2022-10-30 01:00:00,True
4,NW,Lighting,1.4385,1.4385,2022-10-30 00:00:00,True
5,NW,Lighting,1.4385,1.1385,2022-10-30 01:00:00,False
6,NW,Lighting,1.1385,1.4385,2022-10-30 00:00:00,False
7,NW,Lighting,1.1385,1.1385,2022-10-30 01:00:00,True
8,NW,Plugs & Sockets,1431.7830,1431.7830,2022-10-30 00:00:00,True
9,NW,Plugs & Sockets,1431.7830,1404.9005,2022-10-30 01:00:00,False


In [ ]:

# Manteniamo l'indice originale di ogni previsione GBM
unresolved_id = unresolved.reset_index(
    names="gbm_row_id"
)

matches = unresolved_id.merge(
    reference_ambiguous,
    on=["zone", "load_type"],
    how="left",
    suffixes=("_gbm", "_reference")
)

matches["value_match"] = np.isclose(
    matches["y_true"],
    matches["energy_wh_model"],
    atol=0.01,
    rtol=0
)

# Una corrispondenza è univoca solo se identifica
# esattamente un timestamp UTC distinto
summary = (
    matches[matches["value_match"]]
    .groupby("gbm_row_id")
    .agg(
        zone=("zone", "first"),
        load_type=("load_type", "first"),
        unique_utc=("timestamp_utc_reference", "nunique"),
        matching_rows=("timestamp_utc_reference", "size")
    )
)

print("Previsioni ambigue originali:", len(unresolved))
print("Previsioni con corrispondenze:", len(summary))

print("\nRiepilogo:")
display(summary)

print("\nCorrispondenze da controllare:")
display(summary[summary["matching_rows"] != 1])


Previsioni ambigue originali: 20
Previsioni con corrispondenze: 20

Riepilogo:


,zone,load_type,unique_utc,matching_rows
gbm_row_id,,,,
2393,NW,HVAC,1,1
2394,NW,HVAC,1,1
8019,NW,Lighting,1,1
8020,NW,Lighting,1,1
14407,NW,Plugs & Sockets,1,1
14408,NW,Plugs & Sockets,1,1
20796,NE,HVAC,1,1
20797,NE,HVAC,1,1
27210,NE,Plugs & Sockets,1,1



Corrispondenze da controllare:


,zone,load_type,unique_utc,matching_rows
gbm_row_id,,,,
36575,SE,Lighting,2,2
36576,SE,Lighting,2,2


In [ ]:

se_matches = matches[
    (matches["zone"] == "SE") &
    (matches["load_type"] == "Lighting") &
    (matches["value_match"])
].copy()

se_matches["absolute_difference"] = (
    se_matches["y_true"] -
    se_matches["energy_wh_model"]
).abs()

display(
    se_matches[
        ["gbm_row_id", "y_true",
         "energy_wh_model",
         "timestamp_utc_reference",
         "absolute_difference"]
    ].sort_values(
        ["gbm_row_id", "absolute_difference"]
    )
)


,gbm_row_id,y_true,energy_wh_model,timestamp_utc_reference,absolute_difference
24,36575,0.4400,0.4400,2022-10-30 00:00:00,0.0000
25,36575,0.4400,0.4445,2022-10-30 01:00:00,0.0045
27,36576,0.4445,0.4445,2022-10-30 01:00:00,0.0000
26,36576,0.4445,0.4400,2022-10-30 00:00:00,0.0045


In [ ]:

# Cerchiamo soltanto corrispondenze esatte
exact_matches = matches[
    np.isclose(
        matches["y_true"],
        matches["energy_wh_model"],
        atol=1e-6,
        rtol=0
    )
].copy()

# Controlliamo che ogni riga GBM abbia
# esattamente una corrispondenza
counts = exact_matches.groupby("gbm_row_id").size()

assert len(counts) == 20
assert (counts == 1).all()

# Completiamo i timestamp ambigui
gbm_fixed.loc[
    exact_matches["gbm_row_id"],
    "timestamp_utc"
] = pd.to_datetime(
    exact_matches["timestamp_utc_reference"].values
)

print("Total predictions:", len(gbm_fixed))
print("Missing timestamps:",
      gbm_fixed["timestamp_utc"].isna().sum())
print("Corrections applied:", len(exact_matches))


Total predictions: 48221
Missing timestamps: 0
Corrections applied: 20


In [ ]:

# Confrontiamo i timestamp corretti
# con quelli del dataset originale

reference_final = clean[
    ["timestamp_utc", "zone", "load_type", "energy_wh_model"]
].copy()

reference_final["timestamp_utc"] = pd.to_datetime(
    reference_final["timestamp_utc"]
)

verified = gbm_fixed.merge(
    reference_final,
    on=["timestamp_utc", "zone", "load_type"],
    how="left",
    validate="many_to_one"
)

matches_final = np.isclose(
    verified["y_true"],
    verified["energy_wh_model"],
    atol=1e-6,
    rtol=0,
    equal_nan=False
)

print("Original predictions:", len(gbm))
print("Rows after verification:", len(verified))
print("Correct matches:", matches_final.sum())
print("Match percentage:", round(matches_final.mean() * 100, 4))
print("Missing timestamps:", gbm_fixed["timestamp_utc"].isna().sum())
print("Unmatched values:", (~matches_final).sum())


Original predictions: 48221
Rows after verification: 48221
Correct matches: 48221
Match percentage: 100.0
Missing timestamps: 0
Unmatched values: 0


In [ ]:

from pathlib import Path

output_dir = Path("/content/leakage_audit")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "All_Zones_GBM_predictions_utc.csv"

# Manteniamo esattamente le 7 colonne originali
gbm_fixed[gbm.columns].to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("Saved:", output_file)
print("Rows:", len(gbm_fixed))


Saved: /content/leakage_audit/All_Zones_GBM_predictions_utc.csv
Rows: 48221


In [ ]:

from google.colab import files

files.download(
    "/content/leakage_audit/All_Zones_GBM_predictions_utc.csv"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Final Results and Conclusions

### 1. Problem identified

The original `All_Zones_GBM_predictions.csv` file contains timestamps labelled as UTC, although they actually represent Madrid local time.

This issue was previously identified in T39. Incorrect timestamps can cause misalignment between GBM predictions, actual energy consumption and predictions from other models.

### 2. Verification method

To verify the issue, I compared the GBM predictions with the original cleaned dataset, using the zone, load type, timestamp and actual energy consumption (`y_true`).

The results showed:

- **5.28%** of consumption values matched when GBM timestamps were interpreted as UTC.
- **99.96%** matched when timestamps were interpreted as Madrid local time.

This confirmed the timezone issue documented in T39.

### 3. Correction applied

I converted the timestamps from Madrid local time to UTC, accounting for daylight saving time.

During the conversion, 20 timestamps were ambiguous because of the clock change on 30 October 2022.

I resolved these cases by matching the original consumption values with the corresponding UTC timestamps in the cleaned dataset.

No prediction values were modified or removed.

### 4. Final validation

| Validation check | Result |
|---|---:|
| Original predictions | 48,221 |
| Corrected predictions | 48,221 |
| Correct timestamp and consumption matches | 48,221 |
| Match rate | 100% |
| Missing timestamps | 0 |
| Unmatched values | 0 |

The corrected predictions were saved as:

`All_Zones_GBM_predictions_utc.csv`

### 5. Conclusion

The timezone correction was successfully reproduced and independently validated.

All 48,221 GBM predictions are now aligned with the UTC timestamps of the cleaned dataset, without changing the original observed or predicted consumption values.

This makes the corrected file suitable for further model comparison and Hybrid evaluation.

**Important limitation:** This audit verifies timestamp alignment only. It does not resolve data leakage in the GBM input features, which will be investigated separately in `02_GBM_Leakage_Correction.ipynb`.

**Original source:** T39 – `hybrid_blend_learned_weights.ipynb`.